In [ ]:
import pandas as pd

# Load the files
files = ['log5.csv', 'log6.csv', 'log8.csv']
dataframes = []

for f in files:
    df = pd.read_csv(f)
    print(f"File: {f}")
    print(df[['gear', 'mass']].drop_duplicates())
    dataframes.append(df)

# Combine all data to see the full range
all_data = pd.concat(dataframes)
print("\nAll unique gear/mass combinations:")
print(all_data[['gear', 'mass']].drop_duplicates())

In [ ]:
import os

# Check available files
print("Available files:", os.listdir())

# Read first few rows of each
for f in ['log5.csv', 'log6.csv', 'log8.csv']:
    df = pd.read_csv(f)
    print(f"\n{f} gear unique:", df['gear'].unique())
    print(f"{f} mass unique:", df['mass'].unique())

In [ ]:
import glob
import pandas as pd

all_logs = glob.glob('log*.csv')
summary = []
for f in all_logs:
    try:
        df = pd.read_csv(f)
        summary.append({'file': f, 'gear': df['gear'].unique()[0], 'mass': df['mass'].unique()[0]})
    except:
        pass
print(pd.DataFrame(summary))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.interpolate import griddata

# Load specified files
files = ['log5.csv', 'log8.csv', 'log6.csv']
combined_df = pd.concat([pd.read_csv(f) for f in files])

# Extract variables
x = combined_df['o_rpm']  # Output speed
y = combined_df['l_tau']  # Load torque
z = combined_df['eff']    # Efficiency

# Create grid for contouring
xi = np.linspace(x.min(), x.max(), 100)
yi = np.linspace(y.min(), y.max(), 100)
X, Y = np.meshgrid(xi, yi)

# Interpolate efficiency data onto the grid
Z = griddata((x, y), z, (X, Y), method='linear')

# Plot
plt.figure(figsize=(10, 7))
contour = plt.contourf(X, Y, Z, levels=20, cmap='viridis', alpha=0.8)
cbar = plt.colorbar(contour)
cbar.set_label('Efficiency (%)', rotation=270, labelpad=15)

# Overlay the actual data points
for f in files:
    df = pd.read_csv(f)
    plt.scatter(df['o_rpm'], df['l_tau'], marker='o', edgecolors='black', label=f"{f} (Load: {df['mass'].iloc[0]} kg)")

plt.title('Multidimensional Efficiency Map: Output RPM vs. Load Torque')
plt.xlabel('Output Speed (RPM)')
plt.ylabel('Load Torque (N·m)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()

plt.savefig('efficiency_map.png', dpi=300)
print("Efficiency map saved as efficiency_map.png")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.interpolate import griddata

# Load all available log files to get a broad range for the "Map"
all_files = ['log3.csv', 'log5.csv', 'log6.csv', 'log8.csv']
dfs = []
for f in all_files:
    try:
        df = pd.read_csv(f)
        df['source'] = f
        dfs.append(df)
    except:
        pass
combined_df = pd.concat(dfs)

# Coordinates for the Motor Efficiency Map
# Motor Torque is proportional to Current (I_mot)
# Motor Speed is m_rpm
x = combined_df['m_rpm']
y = combined_df['i_mot']
z = combined_df['eff']

# Create grid for the Map (contours)
xi = np.linspace(x.min(), x.max(), 100)
yi = np.linspace(y.min(), y.max(), 100)
X, Y = np.meshgrid(xi, yi)
Z = griddata((x, y), z, (X, Y), method='linear')

# Plot
plt.figure(figsize=(12, 8))
# Draw the efficiency map contours
contour = plt.contourf(X, Y, Z, levels=20, cmap='RdYlGn', alpha=0.7)
cbar = plt.colorbar(contour)
cbar.set_label('System Efficiency (%)', rotation=270, labelpad=15)

# Overlay the operating paths for each file
colors = {'log3.csv': 'red', 'log5.csv': 'blue', 'log6.csv': 'green', 'log8.csv': 'purple'}
labels = {
    'log3.csv': 'Gear 0.6 (75:1), 0.1kg',
    'log5.csv': 'Gear 1.2 (37.5:1), 0.1kg',
    'log6.csv': 'Gear 1.2 (37.5:1), 1.0kg',
    'log8.csv': 'Gear 1.2 (37.5:1), 0.5kg'
}

for f in all_files:
    if f in combined_df['source'].unique():
        df = combined_df[combined_df['source'] == f].sort_values('m_rpm')
        plt.plot(df['m_rpm'], df['i_mot'], color=colors[f], linewidth=2, marker='o', markersize=4, label=labels[f])

plt.title('Multidimensional Efficiency Map (Motor Speed vs. Current)', fontsize=14)
plt.xlabel('Motor Speed (RPM)')
plt.ylabel('Motor Current (Amps)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()

# Annotate Efficiency Sweet Spot
sweet_spot_idx = combined_df['eff'].idxmax()
sweet_spot = combined_df.loc[sweet_spot_idx]
# If there are duplicates in indices, take first
if isinstance(sweet_spot, pd.DataFrame):
    sweet_spot = sweet_spot.iloc[0]

plt.annotate('Max Efficiency Zone', 
             xy=(sweet_spot['m_rpm'], sweet_spot['i_mot']), 
             xytext=(sweet_spot['m_rpm']+100, sweet_spot['i_mot']+0.1),
             arrowprops=dict(facecolor='black', shrink=0.05),
             fontsize=10, fontweight='bold')

plt.savefig('multidimensional_efficiency_map.png', dpi=300)
print("Multidimensional efficiency map saved.")